# Construction Safety Monitor — PPE Retraining **v2** (yolov8m, T4 x2)

Upgrades the PPE model from **yolov8s / 20 epochs (mAP50 ~0.75)** to **yolov8m / 40 epochs** on the same Roboflow PPE Combined v4 dataset (31k train / 8.8k val, 14 classes).

Unlike the smoke retrain, PPE already has plenty of data — so the lever here is **model capacity + more epochs**, not merging datasets. And because old and new use the **same** dataset, the old-vs-new comparison (cell 5) is a valid head-to-head.

### Setup (right-hand panel)
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On** (needed for the Roboflow download)
3. No datasets to attach — the data downloads from Roboflow in cell 1.

### How to run
From a fresh session, **Save Version → Save & Run All (Commit)** to run on both T4s in the background (~8–10h for 40 epochs). Or run cells interactively to sanity-check first.

## 0. Install + GPU check
T4 is Turing (sm_75) — Kaggle's stock torch supports it, so no downgrade needed.

In [ ]:
!pip install -q ultralytics roboflow

import torch
n = torch.cuda.device_count()
print('torch:', torch.__version__, '| GPUs:', n)
for i in range(n):
    print(f'  GPU{i}:', torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), 'No GPU - Settings > Accelerator > GPU T4 x2'
if n < 2:
    print('WARNING: only', n, 'GPU visible - pick "GPU T4 x2", or remove device=[0,1] in the train cell')
print('>>> install OK <<<')

## 1. Download PPE dataset (Roboflow Combined v4)
> SECURITY: the Roboflow key is hardcoded below — keep this notebook **private** and rotate the key after use.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key='toHDg9doTXY2z26D6iYQ')   # personal key - rotate after use
project = rf.workspace('roboflow-universe-projects').project('personal-protective-equipment-combined-model')
ppe = project.version(4).download('yolov8')
PPE_DATA = ppe.location + '/data.yaml'
print('PPE data ->', PPE_DATA)

## 2. Train (yolov8m @ 768px, T4 x2) — tuned to push overall mAP50 > 0.8
768px is the key lever for the small PPE items (goggles/gloves/masks) that drag the average down. Upgrades vs the old model: **yolov8m** (was s), **35 epochs** (was 20), **768px** (was 640), dual-GPU.

⚠️ **Time:** 31k images @ 768px ≈ ~17–22 min/epoch on T4 x2 → 35 epochs ≈ **~11h**, tight against the 12h commit cap. If epoch 1 is slower than ~20 min, drop `EPOCHS` to 30. Realistic target: **~0.80–0.82 overall mAP50** (a few rare classes will still lag — that's expected with 14 classes).

In [ ]:
from ultralytics import YOLO

IMGSZ  = 768          # KEY lever - small PPE items (goggles/gloves/masks) need the resolution
BATCH  = 24           # TOTAL across both T4s; drop to 16 on CUDA OOM at 768px
EPOCHS = 35           # up from the old model's 20
MODEL  = 'yolov8m.pt' # upgrade from the old yolov8s

model = YOLO(MODEL)
model.train(
    data=PPE_DATA,
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, name='ppe_v2', workers=4,
    device=[0, 1],            # both T4 GPUs
    cache=False, patience=15, save=True, val=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    scale=0.5, translate=0.1, fliplr=0.5,
    mosaic=1.0, mixup=0.1, copy_paste=0.0,
    close_mosaic=10, cos_lr=True,
)
print('best ->', 'runs/detect/ppe_v2/weights/best.pt')

## 3. Evaluate (per-class — find the weak PPE classes)
`verbose=True` prints a row per class. Watch the rare ones (Goggles, NO-Goggles, Ladder, Fall-Detected) — those are usually the laggards, and the table is great writeup material.

In [ ]:
from ultralytics import YOLO
r = YOLO('runs/detect/ppe_v2/weights/best.pt').val(data=PPE_DATA, verbose=True)
print('all mAP50:', round(r.box.map50, 3), '| mAP50-95:', round(r.box.map, 3))

## 4. (Optional) Fair comparison vs your current model
PPE old and new use the **same** Roboflow v4 val set, so this is a clean head-to-head (no caveats). Upload your current `models/ppe_model.pt` via **+ Add Input → Upload**, set `OLD`, and run.

In [ ]:
import os
from ultralytics import YOLO

OLD = '/kaggle/input/old-ppe/ppe_model.pt'   # <-- edit to your uploaded path
NEW = 'runs/detect/ppe_v2/weights/best.pt'
if os.path.exists(OLD):
    for tag, w in [('OLD', OLD), ('NEW', NEW)]:
        r = YOLO(w).val(data=PPE_DATA, verbose=False)
        print(f'{tag:4s} | mAP50 {round(r.box.map50,3)} | mAP50-95 {round(r.box.map,3)}')
    print('Same Roboflow v4 val set - this IS a valid head-to-head.')
else:
    print('Upload your current ppe_model.pt (+ Add Input > Upload) and set OLD to its path.')

## 5. Export weights
Copies `best.pt` to `/kaggle/working/ppe_model.pt` (already named for the backend). Download from the **Output** panel, drop into the repo at `models/ppe_model.pt`, restart `uvicorn`. Class IDs/names match the old model, so no code change.

In [ ]:
import shutil
shutil.copy('runs/detect/ppe_v2/weights/best.pt', '/kaggle/working/ppe_model.pt')
print('Saved /kaggle/working/ppe_model.pt - download from the Output panel.')